In [1]:
import h5py as h5
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.models as models
import albumentations as A
from albumentations.pytorch import ToTensorV2
import os
import csv
from torchvision.models import resnet18
from torch.optim.lr_scheduler import StepLR
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from datasets import load_dataset
from PIL import Image
device = 'cuda' if torch.cuda.is_available() else 'cpu'
import warnings 
warnings.filterwarnings("ignore")

/usr/local/lib/python3.10/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.5 (you have 1.4.20). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
from datasets import load_dataset
import h5py
import csv
from PIL import Image

mnist_data = load_dataset("mnist")
train_data = mnist_data["train"]
test_data = mnist_data["test"]
all_images = [np.array(item["image"]) for item in train_data] + [np.array(item["image"]) for item in test_data]
all_labels = [item["label"] for item in train_data] + [item["label"] for item in test_data]
all_images = np.array(all_images)
all_labels = np.array(all_labels)
mask = (all_labels == 1) | (all_labels == 2)
filtered_images = all_images[mask]
filtered_labels = all_labels[mask]
rotation_angles = list(range(0, 360, 30))
n_rotations = len(rotation_angles)
n_original = filtered_images.shape[0]
total_samples = n_original * n_rotations

hdf5_filename = 'mnist_rotated_pair.h5'
csv_filename = 'mnist_rotated_pair_metadata.csv'

with h5py.File(hdf5_filename, 'w') as hdf5_file, open(csv_filename, mode='w', newline='') as csv_file:
    rotated_dataset = hdf5_file.create_dataset('rotated_images', shape=(total_samples, 28, 28), dtype='uint8')
    original_dataset = hdf5_file.create_dataset('original_images', shape=(total_samples, 28, 28), dtype='uint8')
    labels_dataset = hdf5_file.create_dataset('labels', shape=(total_samples,), dtype='uint8')
    angles_dataset = hdf5_file.create_dataset('angle_rotated',shape=(total_samples,),dtype='uint8')
    
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(['sample_index', 'original_index', 'label', 'rotation_angle'])
    
    sample_index = 0
    for orig_idx, (img, label) in enumerate(zip(filtered_images, filtered_labels)):
        pil_img = Image.fromarray(img)
        for angle in rotation_angles:
            rotated_img = pil_img.rotate(angle, resample=Image.BILINEAR, expand=False)
            rotated_array = np.array(rotated_img)
            rotated_dataset[sample_index] = rotated_array
            original_dataset[sample_index] = img
            labels_dataset[sample_index] = label
            angles_dataset[sample_index] = angle
            csv_writer.writerow([sample_index, orig_idx, label, angle])
            sample_index += 1

print(f"Dataset preparation complete. Rotated and original images stored in '{hdf5_filename}' and metadata in '{csv_filename}'.")

README.md:   0%|          | 0.00/6.97k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/15.6M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/2.60M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset preparation complete. Rotated and original images stored in 'mnist_rotated_pair.h5' and metadata in 'mnist_rotated_pair_metadata.csv'.
